In [2]:
# ============================================================
# Block 3: Safe 6-qubit noisy backend setup
# ============================================================

import os

# Thread limits reduce the chance of Aer/Jupyter crashes on Mac.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import random
import inspect

import numpy as np
import torch
import torch.nn as nn

import pennylane as qml
from pennylane import numpy as pnp

import pennylane_qiskit
import qiskit
import qiskit_aer
from qiskit_aer import AerSimulator
import qiskit_ibm_runtime.fake_provider as fake_provider

# ------------------------------------------------------------
# Reproducibility and dtype
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float64)

print("Notebook Python:", sys.executable)
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("PennyLane:", qml.__version__)
print("Qiskit:", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)

# ------------------------------------------------------------
# QNN settings
# ------------------------------------------------------------

n_qubits = 6
shots = 512

# Start shallow for the first noisy run.
n_layers = 1
n_ansatz_layers = 1

print("\nQNN settings:")
print("n_qubits:", n_qubits)
print("shots:", shots)
print("n_layers:", n_layers)
print("n_ansatz_layers:", n_ansatz_layers)

# ------------------------------------------------------------
# Select a fake backend with at least 6 qubits
# ------------------------------------------------------------

candidate_backends = []

for name in dir(fake_provider):
    if not name.startswith("Fake"):
        continue

    obj = getattr(fake_provider, name)

    if not inspect.isclass(obj):
        continue

    try:
        backend = obj()
        num_qubits = getattr(backend, "num_qubits", None)

        if num_qubits is not None and num_qubits >= n_qubits:
            candidate_backends.append((name, backend, num_qubits))

    except Exception:
        pass

candidate_backends = sorted(candidate_backends, key=lambda x: x[2])

print("\nAvailable fake backends with at least 6 qubits:")
for name, _, num_qubits in candidate_backends[:10]:
    print(f"{name}: {num_qubits} qubits")

assert len(candidate_backends) > 0, "No fake backend with at least 6 qubits was found."

backend_class_name, fake_backend, backend_num_qubits = candidate_backends[0]

print("\nSelected fake backend:", backend_class_name)
print("Backend name:", fake_backend.name)
print("Backend qubits:", backend_num_qubits)

# ------------------------------------------------------------
# Create noisy Aer simulator
# ------------------------------------------------------------

aer_backend = AerSimulator.from_backend(fake_backend)

aer_backend.set_options(
    max_parallel_threads=1,
    max_parallel_experiments=1,
    max_parallel_shots=1,
)

print("\nThread-limited Aer backend created.")
print(aer_backend)

# ------------------------------------------------------------
# Create PennyLane qiskit.aer device
# ------------------------------------------------------------

dev_noisy = qml.device(
    "qiskit.aer",
    wires=backend_num_qubits,
    backend=aer_backend,
)

print("\nPennyLane noisy device created.")
print("Device wires:", dev_noisy.wires)

Notebook Python: /Users/mansigoel/Documents/GitHub/Quantum-Machine-Learning-/venv/bin/python
NumPy: 2.2.6
Torch: 2.2.2
PennyLane: 0.45.0
Qiskit: 2.3.0
Qiskit Aer: 0.17.2

QNN settings:
n_qubits: 6
shots: 512
n_layers: 1
n_ansatz_layers: 1

Available fake backends with at least 6 qubits:
FakeCasablancaV2: 7 qubits
FakeJakartaV2: 7 qubits
FakeLagosV2: 7 qubits
FakeNairobiV2: 7 qubits
FakeOslo: 7 qubits
FakePerth: 7 qubits
FakeMelbourneV2: 15 qubits
FakeGuadalupeV2: 16 qubits
FakeAlmadenV2: 20 qubits
FakeBoeblingenV2: 20 qubits

Selected fake backend: FakeCasablancaV2
Backend name: fake_casablanca
Backend qubits: 7

Thread-limited Aer backend created.
AerSimulator('aer_simulator_from(fake_casablanca)'
             noise_model=<NoiseModel on ['id', 'reset', 'x', 'cx', 'measure', 'sx']>)

PennyLane noisy device created.
Device wires: Wires([0, 1, 2, 3, 4, 5, 6])


In [4]:
# ============================================================
# Block 3B: Use fake-backend noise model without coupling-map constraints
# ============================================================

from qiskit_aer.noise import NoiseModel

# ------------------------------------------------------------
# Build a noise model from the selected fake backend
# ------------------------------------------------------------

noise_model = NoiseModel.from_backend(fake_backend)

print("Noise model created from:", fake_backend.name)
print("Noise basis gates:", noise_model.basis_gates)

# ------------------------------------------------------------
# Create Aer simulator with noise, but without hardware coupling constraints
# ------------------------------------------------------------
# Key difference:
#   We are NOT using AerSimulator.from_backend(fake_backend)
# because that also carries device connectivity/coupling constraints.
#
# Instead, we use only the noise model.

aer_backend = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    seed_simulator=SEED,
)

aer_backend.set_options(
    max_parallel_threads=1,
    max_parallel_experiments=1,
    max_parallel_shots=1,
)

print("\nAer simulator created with noise model only.")
print(aer_backend)

# ------------------------------------------------------------
# Recreate PennyLane device
# ------------------------------------------------------------
# Now use exactly n_qubits wires, not all backend wires.
# This avoids weird physical-wire remapping like logical wire 5 -> physical wire 6.

dev_noisy = qml.device(
    "qiskit.aer",
    wires=n_qubits,
    backend=aer_backend,
    basis_gates=noise_model.basis_gates,
    optimization_level=1,
    seed_transpiler=SEED,
)

print("\nPennyLane noisy device recreated.")
print("Device wires:", dev_noisy.wires)
print("Shots will be set on QNode:", shots)

Noise model created from: fake_casablanca
Noise basis gates: ['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']

Aer simulator created with noise model only.
AerSimulator('aer_simulator'
             noise_model=<NoiseModel on ['id', 'reset', 'x', 'cx', 'measure', 'sx']>)

PennyLane noisy device recreated.
Device wires: Wires([0, 1, 2, 3, 4, 5])
Shots will be set on QNode: 512


In [5]:
# ============================================================
# Block 4: Define and test the noisy 6-qubit QNN circuit
# ============================================================

@qml.set_shots(shots)
@qml.qnode(dev_noisy, interface=None, diff_method=None)
def circuit_parallel_noisy(q_params, x):
    """
    Noisy 6-qubit data re-uploading circuit.

    Args:
        q_params:
            Shape = (n_layers, n_ansatz_layers, n_qubits, 3)

        x:
            Shape = (n_qubits,)

    Returns:
        List of 6 expectation values:
            [<Z0>, <Z1>, ..., <Z5>]
    """

    for layer in range(n_layers):

        # Data upload
        for q in range(n_qubits):
            qml.RY(x[q], wires=q)

        # Trainable ansatz
        for ansatz_layer in range(n_ansatz_layers):

            for q in range(n_qubits):
                qml.Rot(
                    q_params[layer, ansatz_layer, q, 0],
                    q_params[layer, ansatz_layer, q, 1],
                    q_params[layer, ansatz_layer, q, 2],
                    wires=q,
                )

            # Logical nearest-neighbour CNOT chain
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q + 1])

    return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]


test_q_params = 0.01 * np.random.randn(
    n_layers,
    n_ansatz_layers,
    n_qubits,
    3,
)

test_x = np.random.randn(n_qubits)

test_q_out = circuit_parallel_noisy(test_q_params, test_x)

print("Noisy QNN test call successful.")
print("test_x shape:", test_x.shape)
print("test_q_params shape:", test_q_params.shape)
print("Quantum output:", test_q_out)
print("Quantum output length:", len(test_q_out))

Noisy QNN test call successful.
test_x shape: (6,)
test_q_params shape: (1, 1, 6, 3)
Quantum output: [array(0.9453125), array(0.90625), array(0.08203125), array(0.046875), array(0.08203125), array(0.05859375)]
Quantum output length: 6


/Users/mansigoel/Documents/GitHub/Quantum-Machine-Learning-/venv/lib/python3.11/site-packages/qiskit/compiler/transpiler.py:269: UserWarning: Providing `coupling_map` and/or `basis_gates` along with `backend` is not recommended, as this will invalidate the backend's gate durations and error rates.
  pm = generate_preset_pass_manager(


In [8]:
# ============================================================
# Block 5: Define Hybrid QNN model without torch.Tensor.numpy()
# ============================================================

class HybridModelNoisy(nn.Module):
    def __init__(self):
        super().__init__()

        # Quantum parameters:
        # shape = (n_layers, n_ansatz_layers, n_qubits, 3)
        self.q_params = nn.Parameter(
            0.01 * torch.randn(
                n_layers,
                n_ansatz_layers,
                n_qubits,
                3,
                dtype=torch.float64,
            )
        )

        # Classical head:
        # takes 6 quantum expectation values and maps to 1 output
        self.classical = nn.Sequential(
            nn.Linear(n_qubits, 1),
            nn.Tanh(),
        ).double()

    def _torch_to_numpy_safe(self, tensor):
        """
        Converts a torch tensor to NumPy without calling tensor.numpy().

        This avoids the PyTorch NumPy bridge issue:
            RuntimeError: Numpy is not available
        """

        if isinstance(tensor, torch.Tensor):
            return np.asarray(
                tensor.detach().cpu().tolist(),
                dtype=np.float64,
            )

        return np.asarray(tensor, dtype=np.float64)

    def quantum_features(self, x_batch, q_params=None):
        """
        Evaluates the noisy QNN one sample at a time.

        Args:
            x_batch:
                torch tensor of shape (batch_size, n_qubits)

            q_params:
                optional q_params candidate.
                Used during SPSA when evaluating theta + cDelta
                and theta - cDelta.

        Returns:
            torch tensor of shape (batch_size, n_qubits)
        """

        if q_params is None:
            q_params = self.q_params

        q_params_np = self._torch_to_numpy_safe(q_params)

        q_outputs = []

        for x in x_batch:
            x_np = self._torch_to_numpy_safe(x)

            q_out = circuit_parallel_noisy(q_params_np, x_np)

            # q_out is a list of scalar arrays.
            q_out_list = [float(v) for v in q_out]

            q_out_tensor = torch.tensor(
                q_out_list,
                dtype=torch.float64,
                device=x_batch.device,
            )

            q_outputs.append(q_out_tensor)

        q_outputs = torch.stack(q_outputs)

        return q_outputs

    def forward(self, x_batch, q_params=None):
        """
        Full hybrid forward pass:

            x_batch -> noisy QNN -> quantum features -> classical head
        """

        q_outputs = self.quantum_features(
            x_batch,
            q_params=q_params,
        )

        # Important:
        # no backprop through noisy quantum circuit
        q_outputs = q_outputs.detach()

        y_pred = self.classical(q_outputs)

        return y_pred.squeeze(-1)

In [9]:
# ============================================================
# Block 6: Test HybridModelNoisy forward pass
# ============================================================

model = HybridModelNoisy().double()

print(model)

batch_size = 3

X_dummy = torch.randn(
    batch_size,
    n_qubits,
    dtype=torch.float64,
)

y_dummy = torch.randn(
    batch_size,
    dtype=torch.float64,
)

y_pred_dummy = model(X_dummy)

print("\nX_dummy shape:", X_dummy.shape)
print("y_dummy shape:", y_dummy.shape)
print("y_pred_dummy shape:", y_pred_dummy.shape)
print("y_pred_dummy:", y_pred_dummy)

HybridModelNoisy(
  (classical): Sequential(
    (0): Linear(in_features=6, out_features=1, bias=True)
    (1): Tanh()
  )
)

X_dummy shape: torch.Size([3, 6])
y_dummy shape: torch.Size([3])
y_pred_dummy shape: torch.Size([3])
y_pred_dummy: tensor([-0.3218, -0.1090, -0.2621], grad_fn=<SqueezeBackward1>)


In [14]:
# ============================================================
# Block 7: Corrected SPSA + Adam training utilities
# ============================================================

# ------------------------------------------------------------
# Loss function
# ------------------------------------------------------------
# Current model ends with Tanh(), so this assumes regression-style
# targets scaled roughly to [-1, 1].
#
# If later you use binary classification, we should remove Tanh()
# and use BCEWithLogitsLoss instead.

loss_fn = nn.MSELoss()


# ------------------------------------------------------------
# PennyLane SPSA optimizer for q_params
# ------------------------------------------------------------

spsa_opt = qml.SPSAOptimizer(
    maxiter=300,
    a=0.02,
    c=0.05,
    A=50,
    alpha=0.602,
    gamma=0.101,
)


# ------------------------------------------------------------
# PyTorch Adam optimizer for classical head
# ------------------------------------------------------------

head_optimizer = torch.optim.Adam(
    model.classical.parameters(),
    lr=1e-3,
)


# ------------------------------------------------------------
# Helper: initialize q_params for PennyLane SPSA
# ------------------------------------------------------------
# We avoid torch_tensor.numpy() because your current PyTorch setup
# had NumPy bridge issues. So we use .tolist().

def init_q_params_for_spsa(model):
    q_params_list = model.q_params.detach().cpu().tolist()

    q_params_pl = pnp.array(
        q_params_list,
        dtype=np.float64,
        requires_grad=True,
    )

    return q_params_pl


q_params_pl = init_q_params_for_spsa(model)

print("Initialized q_params_pl for PennyLane SPSA.")
print("q_params_pl shape:", q_params_pl.shape)


# ------------------------------------------------------------
# Helper: copy PennyLane q_params back into PyTorch model
# ------------------------------------------------------------

def copy_q_params_to_model(model, q_params_pl):
    q_params_np = np.asarray(
        q_params_pl,
        dtype=np.float64,
    )

    q_params_tensor = torch.tensor(
        q_params_np.tolist(),
        dtype=torch.float64,
        device=model.q_params.device,
    )

    with torch.no_grad():
        model.q_params.copy_(q_params_tensor)


# ------------------------------------------------------------
# SPSA cost function factory
# ------------------------------------------------------------
# This creates:
#
#     cost(q_params_candidate) -> PennyLane/NumPy scalar
#
# Important:
#     Do NOT return a Python float.
#     PennyLane SPSA expects the cost value to behave like a
#     NumPy/PennyLane scalar with attributes like .size.

def make_spsa_cost(model, X_batch, y_batch):
    """
    Creates a cost function of q_params only.

    During this cost evaluation:
        - q_params are provided by PennyLane SPSA
        - classical head is held fixed
        - full hybrid loss is evaluated
    """

    X_batch = X_batch.detach()
    y_batch = y_batch.detach()

    def cost(q_params_candidate):
        model.eval()

        with torch.no_grad():
            y_pred = model(
                X_batch,
                q_params=q_params_candidate,
            )

            loss = loss_fn(
                y_pred,
                y_batch,
            )

        # Return PennyLane-compatible scalar, not Python float.
        return pnp.array(
            loss.detach().cpu().item(),
            requires_grad=False,
        )

    return cost


# ------------------------------------------------------------
# One PennyLane SPSA step for q_params
# ------------------------------------------------------------

def pennylane_spsa_step_qparams(
    model,
    q_params_pl,
    X_batch,
    y_batch,
):
    """
    Performs one SPSA update on q_params only.

    Classical head parameters are fixed during this step.
    """

    cost_fn = make_spsa_cost(
        model,
        X_batch,
        y_batch,
    )

    q_params_pl_new = spsa_opt.step(
        cost_fn,
        q_params_pl,
    )

    copy_q_params_to_model(
        model,
        q_params_pl_new,
    )

    return q_params_pl_new


# ------------------------------------------------------------
# One Adam step for classical head
# ------------------------------------------------------------

def adam_step_classical_head(
    model,
    X_batch,
    y_batch,
):
    """
    Performs one Adam update on the classical head only.

    q_params are effectively fixed because quantum features are detached
    inside model.forward().
    """

    model.train()

    head_optimizer.zero_grad()

    y_pred = model(X_batch)

    loss = loss_fn(
        y_pred,
        y_batch,
    )

    loss.backward()

    head_optimizer.step()

    return float(loss.detach().cpu().item())


# ------------------------------------------------------------
# Diagnostic loss evaluation
# ------------------------------------------------------------

def evaluate_hybrid_loss(
    model,
    X_batch,
    y_batch,
):
    model.eval()

    with torch.no_grad():
        y_pred = model(X_batch)

        loss = loss_fn(
            y_pred,
            y_batch,
        )

    return float(loss.detach().cpu().item())


# ------------------------------------------------------------
# Quick test: check SPSA cost output type
# ------------------------------------------------------------

X_test_small = torch.randn(
    1,
    n_qubits,
    dtype=torch.float64,
)

y_test_small = torch.tanh(
    torch.randn(
        1,
        dtype=torch.float64,
    )
)

cost_test = make_spsa_cost(
    model,
    X_test_small,
    y_test_small,
)

cost_value = cost_test(q_params_pl)

print("\nSPSA cost test:")
print("Cost value:", cost_value)
print("Type:", type(cost_value))
print("Shape:", getattr(cost_value, "shape", None))
print("Size:", getattr(cost_value, "size", None))

print("\nCorrected SPSA + Adam utility functions defined.")

Initialized q_params_pl for PennyLane SPSA.
q_params_pl shape: (1, 1, 6, 3)

SPSA cost test:
Cost value: 0.09182305552763947
Type: <class 'pennylane.numpy.tensor.tensor'>
Shape: ()
Size: 1

Corrected SPSA + Adam utility functions defined.


In [15]:
# ============================================================
# Block 8A: Faster one SPSA + Adam sanity check
# ============================================================

num_dummy_samples = 3

X_train_dummy = torch.randn(
    num_dummy_samples,
    n_qubits,
    dtype=torch.float64,
)

y_train_dummy = torch.tanh(
    torch.randn(
        num_dummy_samples,
        dtype=torch.float64,
    )
)

batch_size = 1

idx = torch.randint(
    low=0,
    high=X_train_dummy.shape[0],
    size=(batch_size,),
)

X_batch = X_train_dummy[idx]
y_batch = y_train_dummy[idx]

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)

loss_before = evaluate_hybrid_loss(
    model,
    X_batch,
    y_batch,
)

print("\nLoss before update:", loss_before)

q_norm_before = torch.norm(model.q_params.detach()).item()
head_weight_norm_before = torch.norm(model.classical[0].weight.detach()).item()
head_bias_norm_before = torch.norm(model.classical[0].bias.detach()).item()

print("\nBefore update:")
print("q_params norm:", q_norm_before)
print("head weight norm:", head_weight_norm_before)
print("head bias norm:", head_bias_norm_before)

# ------------------------------------------------------------
# 1. SPSA update for quantum parameters
# ------------------------------------------------------------

print("\nStarting SPSA update...")

q_params_pl = pennylane_spsa_step_qparams(
    model,
    q_params_pl,
    X_batch,
    y_batch,
)

print("SPSA update completed.")

# ------------------------------------------------------------
# 2. Adam update for classical head
# ------------------------------------------------------------

print("\nStarting Adam update...")

head_loss = adam_step_classical_head(
    model,
    X_batch,
    y_batch,
)

print("Adam update completed.")
print("Adam/head loss:", head_loss)

q_norm_after = torch.norm(model.q_params.detach()).item()
head_weight_norm_after = torch.norm(model.classical[0].weight.detach()).item()
head_bias_norm_after = torch.norm(model.classical[0].bias.detach()).item()

print("\nAfter update:")
print("q_params norm:", q_norm_after)
print("head weight norm:", head_weight_norm_after)
print("head bias norm:", head_bias_norm_after)

print("\nChanges:")
print("q_params norm change:", q_norm_after - q_norm_before)
print("head weight norm change:", head_weight_norm_after - head_weight_norm_before)
print("head bias norm change:", head_bias_norm_after - head_bias_norm_before)

X_batch shape: torch.Size([1, 6])
y_batch shape: torch.Size([1])

Loss before update: 0.028064763742764152

Before update:
q_params norm: 0.046874035146278165
head weight norm: 0.6073085754762728
head bias norm: 0.3631889378024384

Starting SPSA update...
SPSA update completed.

Starting Adam update...
Adam update completed.
Adam/head loss: 0.028064763742764152

After update:
q_params norm: 0.046833856164094904
head weight norm: 0.6062245229954901
head bias norm: 0.3641889377720174

Changes:
q_params norm change: -4.017898218326016e-05
head weight norm change: -0.0010840524807826935
head bias norm change: 0.0009999999695790018


In [16]:
# ============================================================
# Block 9: Tiny multi-step SPSA + Adam training loop
# ============================================================

def train_spsa_adam_loop(
    model,
    q_params_pl,
    X_train,
    y_train,
    num_iters=5,
    batch_size=1,
    log_every=1,
):
    """
    Tiny SPSA + Adam training loop.

    Each iteration:
        1. Sample a mini-batch
        2. Update q_params using PennyLane SPSA
        3. Update classical head using PyTorch Adam
        4. Optionally evaluate and log current loss

    Returns:
        updated q_params_pl
        history dictionary
    """

    history = {
        "iter": [],
        "loss": [],
        "q_norm": [],
        "head_weight_norm": [],
        "head_bias_norm": [],
    }

    n_samples = X_train.shape[0]

    for k in range(num_iters):
        # ----------------------------------------------------
        # Sample mini-batch
        # ----------------------------------------------------
        idx = torch.randint(
            low=0,
            high=n_samples,
            size=(batch_size,),
        )

        X_batch = X_train[idx]
        y_batch = y_train[idx]

        # ----------------------------------------------------
        # 1. SPSA update for quantum parameters
        # ----------------------------------------------------
        q_params_pl = pennylane_spsa_step_qparams(
            model,
            q_params_pl,
            X_batch,
            y_batch,
        )

        # ----------------------------------------------------
        # 2. Adam update for classical head
        # ----------------------------------------------------
        head_loss = adam_step_classical_head(
            model,
            X_batch,
            y_batch,
        )

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------
        if k % log_every == 0:
            # Use the same batch for logging to keep it cheap.
            # This is noisy, but good enough for debugging.
            current_loss = evaluate_hybrid_loss(
                model,
                X_batch,
                y_batch,
            )

            q_norm = torch.norm(model.q_params.detach()).item()
            head_weight_norm = torch.norm(model.classical[0].weight.detach()).item()
            head_bias_norm = torch.norm(model.classical[0].bias.detach()).item()

            history["iter"].append(k)
            history["loss"].append(current_loss)
            history["q_norm"].append(q_norm)
            history["head_weight_norm"].append(head_weight_norm)
            history["head_bias_norm"].append(head_bias_norm)

            print(
                f"iter={k:03d} | "
                f"loss={current_loss:.6f} | "
                f"head_loss={head_loss:.6f} | "
                f"q_norm={q_norm:.6f} | "
                f"head_w_norm={head_weight_norm:.6f} | "
                f"head_b_norm={head_bias_norm:.6f}"
            )

    return q_params_pl, history

In [17]:
# ============================================================
# Block 9A: Run tiny dummy SPSA + Adam training loop
# ============================================================

num_dummy_samples = 8

X_train_dummy = torch.randn(
    num_dummy_samples,
    n_qubits,
    dtype=torch.float64,
)

y_train_dummy = torch.tanh(
    torch.randn(
        num_dummy_samples,
        dtype=torch.float64,
    )
)

print("X_train_dummy shape:", X_train_dummy.shape)
print("y_train_dummy shape:", y_train_dummy.shape)

q_params_pl, dummy_history = train_spsa_adam_loop(
    model=model,
    q_params_pl=q_params_pl,
    X_train=X_train_dummy,
    y_train=y_train_dummy,
    num_iters=5,
    batch_size=1,
    log_every=1,
)

print("\nTiny dummy training loop completed.")
print("History keys:", dummy_history.keys())

X_train_dummy shape: torch.Size([8, 6])
y_train_dummy shape: torch.Size([8])
iter=000 | loss=1.360699 | head_loss=1.363730 | q_norm=0.048671 | head_w_norm=0.605225 | head_b_norm=0.363572
iter=001 | loss=1.343503 | head_loss=1.346874 | q_norm=0.051185 | head_w_norm=0.604199 | head_b_norm=0.362784
iter=002 | loss=0.037368 | head_loss=0.038486 | q_norm=0.051088 | head_w_norm=0.603251 | head_b_norm=0.362229
iter=003 | loss=1.674875 | head_loss=1.676480 | q_norm=0.051086 | head_w_norm=0.602706 | head_b_norm=0.361518
iter=004 | loss=0.077407 | head_loss=0.077331 | q_norm=0.050992 | head_w_norm=0.602306 | head_b_norm=0.360959

Tiny dummy training loop completed.
History keys: dict_keys(['iter', 'loss', 'q_norm', 'head_weight_norm', 'head_bias_norm'])
